In [1]:
import os, io, base64, requests, numpy as np
import torch, torch.nn as nn
from PIL import Image

BASE_URL = os.getenv("BASE_URL", "http://154.57.164.81:40306")  # export BASE_URL=

# ---------- model (inlined from the brief) ----------
class CIFAR10CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1); self.bn1 = nn.BatchNorm2d(32)
        self.relu1 = nn.ReLU(); self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1); self.bn2 = nn.BatchNorm2d(64)
        self.relu2 = nn.ReLU(); self.pool2 = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64*8*8, 128); self.relu3 = nn.ReLU()
        self.dropout = nn.Dropout(0.5); self.fc2 = nn.Linear(128, num_classes)
    def forward(self, x):
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout(self.relu3(self.fc1(x)))
        return self.fc2(x)

# ---------- codecs (match the brief exactly) ----------
import torchvision.transforms as T
def tensor_from_b64_png(b64):
    img = Image.open(io.BytesIO(base64.b64decode(b64)))
    if img.size != (32, 32): raise ValueError("Expected 32x32 PNG")
    return T.ToTensor()(img)                       # (3,32,32) in [0,1]

def b64_png_from_tensor(t):
    arr = (t.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)
    buf = io.BytesIO(); Image.fromarray(arr).save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("ascii")

def linf(a, b):
    return float(torch.abs(a - b).max())

# ---------- fetch weights + challenge ----------
open("cifar10_model_best.pth", "wb").write(requests.get(f"{BASE_URL}/model/weights", timeout=30).content)
ck = torch.load("cifar10_model_best.pth", map_location="cpu")
sd = ck["model_state_dict"] if isinstance(ck, dict) and "model_state_dict" in ck else ck
model = CIFAR10CNN().eval(); model.load_state_dict(sd)

ch  = requests.get(f"{BASE_URL}/challenge", timeout=10).json()
x   = tensor_from_b64_png(ch["image"])             # (3,32,32)
tgt = int(ch["target_class"])                      # 3 (cat)
eps = float(ch["epsilon"])                         # 8/255
mean = torch.tensor(ch["normalization"]["mean"]).view(3,1,1)
std  = torch.tensor(ch["normalization"]["std"]).view(3,1,1)

def predict_local(img01):
    with torch.no_grad():
        return int(model(((img01 - mean)/std).unsqueeze(0)).argmax(1))
print("clean pred:", predict_local(x), "target:", tgt, "eps:", round(eps,6))

# ---------- targeted I-FGSM ----------
# normalization is INSIDE the loss path; gradient wrt the [0,1] image.
# step toward target each iter, clip to the eps L-inf ball, keep in [0,1].
def ifgsm(x01, target, eps, steps=100, alpha=None):
    if alpha is None: alpha = eps / 10          # 10x headroom -> rides the ball cleanly
    x0 = x01.clone()
    adv = x01.clone()
    tgt_t = torch.tensor([target])
    for _ in range(steps):
        adv = adv.clone().detach().requires_grad_(True)
        logits = model(((adv - mean)/std).unsqueeze(0))
        loss = nn.functional.cross_entropy(logits, tgt_t)   # minimize -> toward target
        grad = torch.autograd.grad(loss, adv)[0]
        with torch.no_grad():
            adv = adv - alpha * grad.sign()                 # targeted: descend the loss
            adv = torch.max(torch.min(adv, x0 + eps), x0 - eps)  # L-inf projection
            adv = adv.clamp(0.0, 1.0)
    return adv.detach()

adv = ifgsm(x, tgt, eps, steps=100)

# verify against the PNG round-trip (what the server actually scores)
b64 = b64_png_from_tensor(adv)
adv_rt = tensor_from_b64_png(b64)
print(f"after PNG -> pred={predict_local(adv_rt)} (need {tgt})  linf={linf(adv_rt, x):.6f} (<= {eps:.6f})")

# ---------- submit ----------
r = requests.post(f"{BASE_URL}/submit", json={"image": b64}, timeout=10)
print(r.status_code, r.text)

clean pred: 5 target: 3 eps: 0.031373
after PNG -> pred=3 (need 3)  linf=0.031373 (<= 0.031373)
200 {"flag":"[REDACTED]","message":"Congratulations! Challenge solved!","success":true,"validation":{"adversarial_class":"cat","adversarial_prediction":3,"changed_pixel_ratio":0.94921875,"direction_check":true,"gradient_alignment":0.6975911259651184,"linf_constraint":"\u2264 0.031373","linf_norm":0.03137257695198059,"linf_satisfied":true,"sufficient_coverage":true,"target_achieved":true,"target_class":"cat","valid_range":true}}

